# Phase 5: Model Training and Experiment Tracking

This notebook trains popularity-based and personalized SVD recommendation models and tracks experiments using MLflow.

In [3]:
%pip install -q scikit-surprise mlflow

In [5]:
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split

print("Model libraries imported successfully!")

Model libraries imported successfully!


In [6]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np
import pickle
import mlflow

PROJECT_DIR = "/content/drive/MyDrive/recommendation-pipeline"

PROCESSED_DATA_DIR = os.path.join(
    PROJECT_DIR, "data", "processed"
)

FEATURES_DIR = os.path.join(
    PROJECT_DIR, "data", "features"
)

MODEL_DIR = os.path.join(PROJECT_DIR, "models")
REPORT_DIR = os.path.join(PROJECT_DIR, "reports")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

train_df = pd.read_parquet(
    os.path.join(PROCESSED_DATA_DIR, "train_data.parquet")
)

test_df = pd.read_parquet(
    os.path.join(PROCESSED_DATA_DIR, "test_data.parquet")
)

print("Training data:", train_df.shape)
print("Testing data:", test_df.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training data: (80668, 13)
Testing data: (20168, 13)


## Model 1: Popularity-Based Recommendation

In [7]:
# Calculate popularity using training interactions only

popularity_df = train_df.groupby("movieId").agg(
    average_rating=("rating", "mean"),
    rating_count=("rating", "count")
).reset_index()

# Require at least 5 ratings for a movie
popularity_df = popularity_df[
    popularity_df["rating_count"] >= 5
].copy()

# Rank movies by average rating
popularity_df = popularity_df.sort_values(
    by=["average_rating", "rating_count"],
    ascending=[False, False]
).reset_index(drop=True)

print("Popularity model created successfully!")
print("Eligible movies:", len(popularity_df))

display(popularity_df.head(10))

Popularity model created successfully!
Eligible movies: 3123


,movieId,average_rating,rating_count
0,6460,4.900000,5
1,2239,4.666667,6
2,4334,4.600000,5
3,1041,4.590909,11
4,3068,4.583333,6
5,3201,4.571429,7
6,1104,4.552632,19
7,3451,4.545455,11
8,80906,4.500000,10
9,3152,4.500000,8


In [8]:
# save the model
popularity_path = os.path.join(
    MODEL_DIR, "popularity_model.pkl"
)

with open(popularity_path, "wb") as file:
    pickle.dump(popularity_df, file)

print("Popularity model saved successfully!")

Popularity model saved successfully!


## Model 2: Personalized SVD Recommendation

In [9]:
reader = Reader(rating_scale=(0.5, 5.0))

train_data = Dataset.load_from_df(
    train_df[["userId", "movieId", "rating"]],
    reader
)

trainset = train_data.build_full_trainset()

svd_model = SVD(
    n_factors=50,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svd_model.fit(trainset)

print("SVD model training completed successfully!")

SVD model training completed successfully!


In [10]:
# trained model saving
svd_path = os.path.join(MODEL_DIR, "svd_model.pkl")

with open(svd_path, "wb") as file:
    pickle.dump(svd_model, file)

print("SVD model saved successfully!")

SVD model saved successfully!


In [11]:
# Evaluating the SVD model using RMSE and MAE
testset = list(
    test_df[["userId", "movieId", "rating"]]
    .itertuples(index=False, name=None)
)

svd_predictions = svd_model.test(testset)

svd_rmse = accuracy.rmse(
    svd_predictions, verbose=False
)

svd_mae = accuracy.mae(
    svd_predictions, verbose=False
)

print("SVD RMSE:", round(svd_rmse, 4))
print("SVD MAE:", round(svd_mae, 4))

SVD RMSE: 1.0145
SVD MAE: 0.7827


## Experiment Tracking with MLflow

In [13]:
import os
import mlflow

# Create permanent storage locations in Google Drive
MLFLOW_DB = os.path.join(PROJECT_DIR, "mlflow.db")
MLFLOW_ARTIFACTS = os.path.join(PROJECT_DIR, "mlartifacts")

os.makedirs(MLFLOW_ARTIFACTS, exist_ok=True)

# Configure MLflow to use SQLite
mlflow.set_tracking_uri(
    "sqlite:////content/drive/MyDrive/recommendation-pipeline/mlflow.db"
)

experiment_name = "MovieLens_Recommendation_System"

# Create experiment if it does not already exist
existing_experiment = mlflow.get_experiment_by_name(
    experiment_name
)

if existing_experiment is None:
    mlflow.create_experiment(
        experiment_name,
        artifact_location=MLFLOW_ARTIFACTS
    )

mlflow.set_experiment(experiment_name)

print("MLflow configured successfully!")
print("Tracking database:", MLFLOW_DB)
print("Artifact directory:", MLFLOW_ARTIFACTS)

2026/11/14 00:19:19 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/14 00:19:20 INFO mlflow.store.db.utils: Updating database tables


MLflow configured successfully!
Tracking database: /content/drive/MyDrive/recommendation-pipeline/mlflow.db
Artifact directory: /content/drive/MyDrive/recommendation-pipeline/mlartifacts


In [14]:
with mlflow.start_run(run_name="SVD_Model"):

    mlflow.log_param("model", "SVD")
    mlflow.log_param("n_factors", 50)
    mlflow.log_param("n_epochs", 20)
    mlflow.log_param("lr_all", 0.005)
    mlflow.log_param("reg_all", 0.02)

    mlflow.log_metric("RMSE", svd_rmse)
    mlflow.log_metric("MAE", svd_mae)

    mlflow.log_artifact(svd_path)

print("SVD experiment logged successfully!")

SVD experiment logged successfully!


In [15]:
experiment = mlflow.get_experiment_by_name(
    "MovieLens_Recommendation_System"
)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

print("Experiment name:", experiment.name)
print("Number of recorded runs:", len(runs))

display(
    runs[["run_id", "status", "metrics.RMSE", "metrics.MAE"]]
)

Experiment name: MovieLens_Recommendation_System
Number of recorded runs: 1


,run_id,status,metrics.RMSE,metrics.MAE
0,1c709f03d9b140ceb74cafc1f744134d,FINISHED,1.014546,0.782679
